# Learning Buddy - server penelitian di Kaggle

Menyalakan sistem lengkap (Qwen2.5-7B-Instruct 4-bit, 12 adapter jenjang, penyandi/FAISS,
cross-encoder, dan empat agen) lalu membukanya lewat **Cloudflare Quick Tunnel** yang gratis
dan tanpa akun. UI dijalankan di laptop dan menyambung ke sini.

**Persiapan**

1. Input (tiga-tiganya):
   - Dataset **learning-buddy-kode** (kode + potongan buku);
   - Dataset **adapter-pjok** (isi folder `D:\Skripsi\adapter-pjok\adapters_recursive`,
     adapter PJOK kelas 1-6);
   - output notebook pelatihan (*Add Input > Your Work*), berisi `adapters/PKN/` dan
     `hasil/ambang_relevan.json`.
2. *Settings*: Accelerator **GPU T4 x2**, Internet **On**.
3. *Add-ons > Secrets*: `TB_KUNCI` (kata sandi bebas untuk membatasi pemakai tautan).

Jalankan sel satu per satu atau **Run All** (sesi interaktif, bukan Commit). Sel tunnel
mencetak "Server GPU aktif di : https://...trycloudflare.com"; alamat itu dipakai di VS Code
(konfigurasi "3. Gerbang ke GPU Kaggle").

In [ ]:
# Salin kode ke folder kerja
import glob, os, shutil, subprocess, sys, time, re, json

# Matikan sisa server/tunnel dari jalankan sebelumnya. Proses latar itu TIDAK ikut mati saat
# kernel di-restart dan tetap memegang memori GPU (penyebab CUDA out of memory).
for nama in ('tb.server', 'cloudflared'):
    subprocess.run(['pkill', '-f', nama], check=False)
time.sleep(3)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
# indeks FAISS disimpan di luar folder kerja agar tidak dibangun ulang setiap Run All
os.environ.setdefault('TB_FAISS', '/kaggle/working/faiss_cache')
print(subprocess.run(['nvidia-smi', '--query-gpu=index,memory.used,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)

# dicari di mana pun dalam input: zip, folder LearningBuddy, atau isi foldernya langsung
hit = glob.glob('/kaggle/input/**/tb/agen.py', recursive=True)
assert hit, 'Dataset learning-buddy-kode belum ditambahkan sebagai input.'
# Bila ada lebih dari satu salinan kode (dataset lama ikut terpasang), pakai yang terbaru:
# yang sudah punya tb/validasi_recursive.py, lalu yang berkasnya paling baru.
calon = [h.rsplit('/tb/', 1)[0] for h in hit]
calon.sort(key=lambda d: (os.path.exists(d + '/tb/validasi_recursive.py'),
                          os.path.getmtime(d + '/tb/server.py')), reverse=True)
if len(calon) > 1:
    print('PERINGATAN: ada', len(calon), 'salinan kode di input; dipakai yang terbaru.')
    print('  Sebaiknya lepas dataset kode lama dari Input:', calon[1:])
SUMBER = calon[0]
assert os.path.exists(SUMBER + '/data/potongan.jsonl'),     'Folder data/ tidak ikut terunggah; unggah ulang seluruh folder LearningBuddy.'
KERJA = '/kaggle/working/LearningBuddy'
shutil.rmtree(KERJA, ignore_errors=True)
shutil.copytree(SUMBER, KERJA)
os.chdir(KERJA)
sys.path.insert(0, KERJA)
print('kode:', SUMBER, '->', KERJA)

In [ ]:
# Temukan adapter di semua input: output notebook pelatihan (adapters/PKN/grade_k) dan
# dataset adapter PJOK (mis. adapter-pjok/adapters_recursive/grade_k), apa pun nama foldernya.
SEMUA_ADAPTER = sorted(glob.glob('/kaggle/input/**/grade_1/adapter_config.json', recursive=True))
print('folder adapter yang ditemukan:', *SEMUA_ADAPTER, sep='\n  ')

def pola_adapter(mapel):
    lain = 'pkn' if mapel == 'PJOK' else 'pjok'
    def cocok(p):
        q = p.lower()
        if f'/{lain}/' in q or f'adapter-{lain}' in q:
            return False
        return f'/{mapel.lower()}/' in q or f'adapter-{mapel.lower()}' in q or \
            (mapel == 'PJOK' and 'adapters_recursive' in q)
    c = [p for p in SEMUA_ADAPTER if cocok(p)]
    return os.path.dirname(c[0])[:-1] + '{k}' if c else None

ADAPTER = {m: pola_adapter(m) for m in ('PJOK', 'PKN')}
for m, p in ADAPTER.items():
    n = sum(os.path.exists(p.format(k=k)) for k in range(1, 7)) if p else 0
    print(f'adapter {m}: {n}/6  {p or "(tidak ada - model dasar dipakai)"}')

In [ ]:
# Pasang pustaka. Versi dicetak agar dapat dicatat di Bab IV.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
                '-r', 'requirements-gpu.txt'], check=True)
import importlib, importlib.metadata as md
importlib.invalidate_caches()
for p in ['torch', 'transformers', 'peft', 'bitsandbytes', 'accelerate',
          'sentence-transformers', 'faiss-cpu', 'langgraph', 'fastapi']:
    try: print(f'{p:22s}', md.version(p))
    except Exception: print(f'{p:22s} TIDAK ADA')
import torch
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Rahasia
os.environ['TB_KUNCI'] = ''
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for nama in ['TB_KUNCI', 'HF_TOKEN']:
        try: os.environ[nama] = s.get_secret(nama)
        except Exception: pass
except Exception:
    pass
print('kunci akses aktif' if os.environ.get('TB_KUNCI') else
      'PERINGATAN: TB_KUNCI kosong, siapa pun yang tahu tautannya dapat memakai GPU ini')

In [ ]:
# Indeks FAISS potongan Recursive (sudah dirapikan) per mapel per kelas, buku Kurikulum
# Merdeka, dengan penyandi pilihan (4.379 potongan, sekitar 5-8 menit).
# Ganti TB_MODEL_EMBEDDING bila hasil tb.uji_embedding memilih penyandi lain.
os.environ.setdefault('TB_MODEL_EMBEDDING', 'Qwen/Qwen3-Embedding-4B')
os.environ.setdefault('TB_DEVICE_CARI', 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda')
print('penyandi', os.environ['TB_MODEL_EMBEDDING'], 'di', os.environ['TB_DEVICE_CARI'])
assert torch.cuda.device_count() > 1, 'Aktifkan Accelerator GPU T4 x2: penyandi 4B butuh GPU kedua'
from tb.pencarian import muat_potongan, bangun_indeks_faiss, folder_indeks, boleh_dicari, MODEL_EMBEDDING
ps = muat_potongan()
meta = os.path.join(folder_indeks(os.environ['TB_MODEL_EMBEDDING']), 'meta.json')
lama = json.load(open(meta)) if os.path.exists(meta) else {}
# dipakai ulang hanya bila indeks tersimpan dibangun dari potongan yang sama
if 'strategi' in lama and lama.get('potongan') == sum(boleh_dicari(p) for p in ps):
    print('indeks sudah ada, dipakai ulang:', lama)
else:
    bangun_indeks_faiss(ps, os.environ['TB_MODEL_EMBEDDING'])
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Ambang pengecekan 1 (apakah potongan buku memang membahas pertanyaannya), hasil kalibrasi
# notebook pelatihan. Bila tidak ada, kalibrasi di sini (sekitar 5-10 menit).
tujuan = os.path.join(folder_indeks(os.environ['TB_MODEL_EMBEDDING']), 'ambang_relevan.json')
# hanya ambang hasil kalibrasi korpus Merdeka (ambang lama dibuat dengan korpus campuran)
ada = [p for p in glob.glob('/kaggle/input/**/hasil/ambang_relevan.json', recursive=True)
       if 'Merdeka' in json.load(open(p)).get('metode', '')]
if ada:
    shutil.copy(ada[0], tujuan)
elif not os.path.exists(tujuan):
    from tb.evaluasi_pencarian import evaluasi
    evaluasi()
    os.makedirs('/kaggle/working/hasil', exist_ok=True)       # unduh dari panel Output
    for f in ['evaluasi_pencarian.json', 'Evaluasi_Pencarian.xlsx', 'Validasi_Soal_PJOK_Recursive.xlsx']:
        shutil.copy(f'hasil/{f}', f'/kaggle/working/hasil/{f}')
    import gc; gc.collect(); torch.cuda.empty_cache()
print('ambang pengecekan 1:', json.load(open(tujuan))['ambang'])

In [ ]:
# Nyalakan server di latar belakang (memuat Qwen 7B dan 12 adapter, sekitar 5-8 menit)
import requests
perintah = [sys.executable, '-m', 'tb.server', '--qwen', '--host', '0.0.0.0', '--port', '8000',
            '--cari', 'semantik']
if ADAPTER['PJOK']: perintah += ['--adapter-pjok', ADAPTER['PJOK']]
if ADAPTER['PKN']:  perintah += ['--adapter-pkn', ADAPTER['PKN']]
log = open('server.log', 'w')
server = subprocess.Popen(perintah, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

for i in range(120):
    time.sleep(10)
    if server.poll() is not None:
        print(open('server.log').read()[-4000:])
        raise RuntimeError('server berhenti, lihat log di atas')
    try:
        s = requests.get('http://127.0.0.1:8000/api/sehat', timeout=5).json()
        print('SIAP:', json.dumps(s, indent=1))
        break
    except Exception:
        print(f'menunggu... {10 * (i + 1)} detik', end='\r')

In [ ]:
# Buka ke internet lewat Cloudflare Quick Tunnel (gratis, tanpa akun)
if not os.path.exists('/kaggle/working/cloudflared'):
    subprocess.run(['wget', '-q', '-O', '/kaggle/working/cloudflared',
                    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'],
                   check=True)
    os.chmod('/kaggle/working/cloudflared', 0o755)
tlog = open('tunnel.log', 'w')
tunnel = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--no-autoupdate',
                           '--url', 'http://127.0.0.1:8000'], stdout=tlog, stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m:
        URL = m.group(0); break
print('=' * 78)
print('Server GPU aktif di :', URL)
print()
print('Tempel di terminal VS Code laptop (folder LearningBuddy):')
print(f'   python -m tb.server --remote {URL} --remote-kunci <isi TB_KUNCI Anda>')
print('lalu buka http://127.0.0.1:8000 (chat) dan http://127.0.0.1:8000/lab.html (lab)')
print('=' * 78)

In [ ]:
# Uji keempat agen pada kedua mata pelajaran, termasuk pagar pengaruh adapter
h = {'X-Kunci': os.environ.get('TB_KUNCI', '')}
def tanya(q, mapel='PJOK', kelas=None, skenario='S4'):
    return requests.post('http://127.0.0.1:8000/api/tanya', headers=h, timeout=600,
                         json={'pertanyaan': q, 'mata_pelajaran': mapel,
                               'kelas': kelas, 'skenario': skenario}).json()

for q, m in [('napa hrs pemanasan dlu sblm olhraga?', 'PJOK'),
             ('apa arti gotong royong', 'PKN'), ('halo', 'PJOK')]:
    r = tanya(q, m)
    print(f'[{m}] {q}')
    print('  Perceive :', r['pertanyaan_bersih'], '| jelas', r['jelas'])
    print('  Plan     :', 'kelas', r['kelas'], r['sumber_kelas'], r['keyakinan_kelas'], '| topik', r['topik'])
    print('  Act      :', r['putaran'], 'putaran | adapter', r['adapter_aktif'],
          '| konteks', r['konteks'] and r['konteks']['id'])
    print('  Reflect  :', r['status'], r['ukuran'] and (r['ukuran']['pkr'], r['ukuran']['kld']))
    print('  jawaban  :', r['jawaban'], '\n')

for m, q in [('PJOK', 'Apa manfaat pemanasan?'), ('PKN', 'Apa arti musyawarah?')]:
    a = tanya(q, m, kelas=4, skenario='S1')['riwayat'][0]['jawaban']
    b = tanya(q, m, kelas=4, skenario='S2')['riwayat'][0]['jawaban']
    print(f'{m} S1: {a}\n{m} S2: {b}')
    print('  PERINGATAN: adapter tidak mengubah keluaran' if a == b else '  adapter berpengaruh')

In [ ]:
# Penjaga sesi. Biarkan sel ini berjalan selama aplikasi dipakai.
try:
    while True:
        time.sleep(300)
        hidup = server.poll() is None and tunnel.poll() is None
        jejak = sorted(glob.glob('log/jejak_*.jsonl'))
        n = sum(1 for _ in open(jejak[-1])) if jejak else 0
        print(time.strftime('%H:%M'), 'server', 'hidup' if hidup else 'MATI', '| pertanyaan hari ini:', n)
        if not hidup:
            print(open('server.log').read()[-3000:]); break
except KeyboardInterrupt:
    print('dihentikan')

**Catatan**

- Tautan `trycloudflare.com` berganti setiap kali notebook dijalankan ulang.
- Jejak setiap pertanyaan tersimpan di `/kaggle/working/LearningBuddy/log/`.
- Kuota GPU Kaggle 30 jam per minggu. Matikan sesi (*Stop Session*) bila tidak dipakai.